In [39]:
# =============================================================================
# 1. DATA PREPROCESSING AND FEATURE ENGINEERING FOR ML MODELS
# =============================================================================

print("="*80)
print("MACHINE LEARNING MODELS IMPLEMENTATION")
print("="*80)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

# Advanced ML Libraries
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Time series libraries
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

print("Libraries imported successfully!")
print(f"LightGBM version: {lgb.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"CatBoost version: {cb.__version__}")

# Load the processed data
print("\nLoading processed data...")
refactored_df = pd.read_csv('../data/refactored_df.csv')
weather_data = pd.read_csv('../data/weather_data.csv')
trends_df = pd.read_csv('../data/customer_trends.csv')

# Convert date columns
refactored_df['Date'] = pd.to_datetime(refactored_df['Date'])
refactored_df['Branch'].replace({'SBD1': 'VJW'}, inplace=True)
weather_data['Date'] = pd.to_datetime(weather_data['Date'])
trends_df['Date'] = pd.to_datetime(trends_df['Month'])

print(f"Data loaded successfully!")
print(f"Sales data shape: {refactored_df.shape}")
print(f"Weather data shape: {weather_data.shape}")
print(f"Trends data shape: {trends_df.shape}")


MACHINE LEARNING MODELS IMPLEMENTATION
Libraries imported successfully!
LightGBM version: 4.6.0
XGBoost version: 3.1.1
CatBoost version: 1.2.8

Loading processed data...
Data loaded successfully!
Sales data shape: (147595, 9)
Weather data shape: (456, 11)
Trends data shape: (82, 3)


In [40]:
main_df = (
    refactored_df.groupby([refactored_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Qty': 'sum'})
    .reset_index()
)
main_df.columns = ['Date', 'Branch', 'Qty']
main_df = main_df.merge(weather_data, on=['Date', 'Branch']).reset_index(drop=True)
main_df = main_df.merge(trends_df, on=['Date'])
main_df = main_df.drop(columns=['Month'])
main_df['Seasonality_Level'] = main_df['Date'].dt.month_name().str[:3].map({
    'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
    'May': 2, 'Jan': 2,
    'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
})

In [43]:
# =============================================================================
# 2. COMPREHENSIVE FEATURE ENGINEERING
# =============================================================================

from sklearn.preprocessing import LabelEncoder
print("\n2. COMPREHENSIVE FEATURE ENGINEERING")
print("-" * 50)

def drop_nan_columns(df, threshold=1.0, verbose=True):
    """
    Drop columns with a fraction of NaN values above the given threshold.
    """
    nan_ratio = df.isna().mean()
    drop_cols = nan_ratio[nan_ratio >= threshold].index.tolist()
    df_cleaned = df.drop(columns=drop_cols)

    if verbose:
        print(
            f"Dropped {len(drop_cols)} column(s) with ≥{threshold*100:.0f}% NaN values:")
        if drop_cols:
            print(drop_cols)
        else:
            print("No columns dropped.")

    return df_cleaned


def group_expanding_shifted_mean(group):
    """Helper function for expanding mean with shift - reduces overhead when reused"""
    return group.expanding().mean().shift(1)


def group_rolling_shifted_mean(group, window):
    """Helper function for rolling mean with shift - for complete past-only purity"""
    return group.shift(1).rolling(window=window, min_periods=1).mean()


def create_ml_features_leakage_safe(df):
    """
    Create comprehensive features for machine learning models with COMPLETE LEAKAGE-SAFE engineering
    """
    print("Creating comprehensive features with COMPLETE leakage-safe engineering...")

    # Start with the base dataframe
    ml_df = df.copy()
    ml_df = drop_nan_columns(ml_df, 0.5)

    # Check data availability for each branch
    branch_counts = ml_df.groupby('Branch').size()
    print(f"  Data points per branch: {dict(branch_counts)}")

    # CRITICAL: Sort by date and branch FIRST for proper temporal ordering
    ml_df = ml_df.sort_values(['Branch', 'Date']).reset_index(drop=True)

    # Cache grouped objects for efficiency
    branch_qty_group = ml_df.groupby('Branch')['Qty']
    branch_temp_group = ml_df.groupby('Branch')['Avg Temp']
    branch_humidity_group = ml_df.groupby('Branch')['Avg Humidity']
    branch_wind_group = ml_df.groupby('Branch')['Avg Wind Speed']

    # 1. TIME-BASED FEATURES
    print("  Creating time-based features...")
    ml_df['year'] = ml_df['Date'].dt.year
    ml_df['month'] = ml_df['Date'].dt.month
    ml_df['day'] = ml_df['Date'].dt.day
    ml_df['dayofweek'] = ml_df['Date'].dt.dayofweek
    ml_df['dayofyear'] = ml_df['Date'].dt.dayofyear
    ml_df['week'] = ml_df['Date'].dt.isocalendar().week.astype(int)
    ml_df['quarter'] = ml_df['Date'].dt.quarter

    # Cyclical encoding for time features
    ml_df['month_sin'] = np.sin(2 * np.pi * ml_df['month'] / 12)
    ml_df['month_cos'] = np.cos(2 * np.pi * ml_df['month'] / 12)
    ml_df['dayofweek_sin'] = np.sin(2 * np.pi * ml_df['dayofweek'] / 7)
    ml_df['dayofweek_cos'] = np.cos(2 * np.pi * ml_df['dayofweek'] / 7)
    ml_df['quarter_sin'] = np.sin(2 * np.pi * ml_df['quarter'] / 4)
    ml_df['quarter_cos'] = np.cos(2 * np.pi * ml_df['quarter'] / 4)

    # 2. LAG FEATURES (LEAKAGE-SAFE)
    print("  Creating lag features...")
    lag_periods = [1, 2, 3, 6, 12]  # months instead of days
    for lag in lag_periods:
        ml_df[f'qty_lag_{lag}m'] = branch_qty_group.shift(lag)
        # FIXED: Use groupby().apply() for proper branch boundaries
        if lag <= 3:  # Only for short lags
            lag_val = lag  # Capture loop variable
            ml_df[f'qty_lag_{lag}m_mean'] = (
                branch_qty_group
                .apply(lambda x: x.shift(lag_val).rolling(window=min(3, lag_val), min_periods=1).mean())
                .reset_index(level=0, drop=True)
            )

    # 3. ROLLING STATISTICS (LEAKAGE-SAFE)
    print("  Creating rolling statistics...")
    rolling_windows = [2, 3, 6, 12]  # months
    for window in rolling_windows:
        # FIXED: Use groupby().apply() for proper branch boundaries
        # OPTIONAL: For complete past-only purity, uncomment the shift(1) version
        window_val = window  # Capture loop variable
        ml_df[f'qty_rolling_mean_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )
        # For complete past-only: .apply(lambda x: x.shift(1).rolling(window=window_val, min_periods=1).mean())

        ml_df[f'qty_rolling_std_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).std())
            .reset_index(level=0, drop=True)
        )
        ml_df[f'qty_rolling_max_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).max())
            .reset_index(level=0, drop=True)
        )
        ml_df[f'qty_rolling_min_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).min())
            .reset_index(level=0, drop=True)
        )
        ml_df[f'qty_rolling_sum_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).sum())
            .reset_index(level=0, drop=True)
        )

    # 4. EXPANDING STATISTICS (LEAKAGE-SAFE - SHIFT INSIDE GROUP)
    print("  Creating expanding statistics...")
    # FIXED: Use helper function for efficiency
    ml_df['qty_expanding_mean'] = (
        branch_qty_group
        .apply(group_expanding_shifted_mean)
        .reset_index(level=0, drop=True)
    )
    ml_df['qty_expanding_std'] = (
        branch_qty_group
        .apply(lambda x: x.expanding().std().shift(1))
        .reset_index(level=0, drop=True)
    )
    ml_df['qty_expanding_max'] = (
        branch_qty_group
        .apply(lambda x: x.expanding().max().shift(1))
        .reset_index(level=0, drop=True)
    )
    ml_df['qty_expanding_min'] = (
        branch_qty_group
        .apply(lambda x: x.expanding().min().shift(1))
        .reset_index(level=0, drop=True)
    )

    # OPTIONAL ENHANCEMENTS: Additional useful features
    print("  Creating additional enhancement features...")
    # Rate of change (lag diff)
    ml_df['qty_diff_1m'] = branch_qty_group.diff(1)
    ml_df['qty_diff_3m'] = branch_qty_group.diff(3)

    # Previous year same month lag
    ml_df['qty_lag_12m'] = branch_qty_group.shift(12)

    # Sales volatility (rolling std shifted)
    ml_df['qty_volatility_3m'] = (
        branch_qty_group
        .apply(lambda x: x.rolling(3).std().shift(1))
        .reset_index(level=0, drop=True)
    )

    # 5. SEASONAL FEATURES (LEAKAGE-SAFE - PER-BRANCH-PER-MONTH)
    print("  Creating leakage-safe seasonal features...")
    # FIXED: Compute per-branch-per-month expanding means
    ml_df['monthly_seasonality'] = (
        ml_df.groupby(['Branch', 'month'])['Qty']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )

    ml_df['quarterly_seasonality'] = (
        ml_df.groupby(['Branch', 'quarter'])['Qty']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )

    ml_df['dow_seasonality'] = (
        ml_df.groupby(['Branch', 'dayofweek'])['Qty']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )

    # 6. WEATHER LAG FEATURES (LEAKAGE-SAFE)
    print("  Creating weather lag features...")
    weather_lags = [1, 2, 3, 6]  # months
    for lag in weather_lags:
        ml_df[f'temp_lag_{lag}m'] = branch_temp_group.shift(lag)
        ml_df[f'humidity_lag_{lag}m'] = branch_humidity_group.shift(lag)
        ml_df[f'wind_lag_{lag}m'] = branch_wind_group.shift(lag)

    # 7. TRENDS LAG FEATURES (LEAKAGE-SAFE - GROUP BY BRANCH IF NEEDED)
    print("  Creating trends lag features...")
    trends_lags = [1, 2, 3, 6]
    for lag in trends_lags:
        # Check if Interest varies by Branch
        if ml_df.groupby('Branch')['Interest'].nunique().max() > 1:
            # Interest varies by branch, so group by Branch
            ml_df[f'trends_lag_{lag}m'] = ml_df.groupby(
                'Branch')['Interest'].shift(lag)
        else:
            # Interest is global, so shift globally
            ml_df[f'trends_lag_{lag}m'] = ml_df['Interest'].shift(lag)

    # 8. PRODUCT FEATURES (LABEL ENCODER MOVED TO AFTER SPLIT)
    print("  Creating product features...")
    # NOTE: LabelEncoder will be fitted after train/test split to avoid leakage
    # For now, just create a placeholder
    ml_df['branch_encoded'] = 0  # Will be properly encoded after split

    # 9. INTERACTION FEATURES
    print("  Creating interaction features...")
    ml_df['temp_humidity_interaction'] = ml_df['Avg Temp'] * \
        ml_df['Avg Humidity']
    ml_df['temp_wind_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Wind Speed']

    # 10. STATISTICAL FEATURES (LEAKAGE-SAFE)
    print("  Creating leakage-safe statistical features...")
    # Temperature statistics
    ml_df['temp_range'] = ml_df['Max Temp'] - ml_df['Min Temp']
    ml_df['humidity_range'] = ml_df['Max Humidity'] - ml_df['Min Humidity']
    ml_df['wind_range'] = ml_df['Max Wind Speed'] - ml_df['Min Wind Speed']

    # FIXED: Temperature deviation using expanding mean + shift per branch-month
    ml_df['temp_monthly_mean_past'] = (
        ml_df.groupby(['Branch', 'month'])['Avg Temp']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    ml_df['temp_deviation'] = ml_df['Avg Temp'] - \
        ml_df['temp_monthly_mean_past']

    # FIXED: Humidity deviation using expanding mean + shift per branch-month
    ml_df['humidity_monthly_mean_past'] = (
        ml_df.groupby(['Branch', 'month'])['Avg Humidity']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    ml_df['humidity_deviation'] = ml_df['Avg Humidity'] - \
        ml_df['humidity_monthly_mean_past']

    # 11. BUSINESS FEATURES (LEAKAGE-SAFE)
    print("  Creating leakage-safe business features...")
    # Days since last sale (convert to months for monthly data)
    ml_df['months_since_last_sale'] = ml_df.groupby(
        'Branch')['Date'].diff().dt.days / 30.44

    # FIXED: Sales momentum using shifted expanding mean
    ml_df['sales_momentum_3m'] = np.where(
        ml_df['qty_expanding_mean'] != 0,
        ml_df['qty_rolling_mean_3m'] / ml_df['qty_expanding_mean'],
        1.0
    )
    ml_df['sales_momentum_6m'] = np.where(
        ml_df['qty_expanding_mean'] != 0,
        ml_df['qty_rolling_mean_6m'] / ml_df['qty_expanding_mean'],
        1.0
    )

    # FIXED: Market share using proper denominator alignment
    # Compute total quantity per Date first
    date_totals = ml_df.groupby(
        'Date')['Qty'].sum().reset_index(name='total_qty')

    # Merge and shift total_qty globally by chronological order
    date_totals = date_totals.sort_values('Date')
    date_totals['total_qty_prev'] = date_totals['total_qty'].shift(1)

    # Merge back into ml_df
    ml_df = ml_df.merge(
        date_totals[['Date', 'total_qty_prev']], on='Date', how='left')
    ml_df['prev_branch_qty'] = branch_qty_group.shift(1)
    ml_df['prev_market_share'] = np.where(
        ml_df['total_qty_prev'] > 0,
        ml_df['prev_branch_qty'] / ml_df['total_qty_prev'],
        0
    )
    # Clean up temporary columns
    ml_df.drop(['total_qty_prev', 'prev_branch_qty'], axis=1, inplace=True)

    # 12. HANDLE REMAINING NaN VALUES (OPTIMIZED)
    print("  Handling remaining NaN values...")

    # FIXED: Vectorized NaN filling for efficiency with safety checks
    fill_dict = {}

    # Safe median calculation
    qty_median = ml_df['Qty'].median() if 'Qty' in ml_df.columns else 0

    # Fill lag features with 0
    lag_cols = [col for col in ml_df.columns if 'lag' in col]
    for col in lag_cols:
        fill_dict[col] = 0

    # Fill rolling features
    rolling_cols = [col for col in ml_df.columns if 'rolling' in col]
    for col in rolling_cols:
        if col.endswith('_mean') or col.endswith('_sum'):
            fill_dict[col] = qty_median
        else:
            fill_dict[col] = ml_df[col].median() if col in ml_df.columns else 0

    # Fill expanding features
    expanding_cols = [col for col in ml_df.columns if 'expanding' in col]
    for col in expanding_cols:
        fill_dict[col] = qty_median

    # Fill seasonal features
    seasonal_cols = [col for col in ml_df.columns if 'seasonality' in col]
    for col in seasonal_cols:
        fill_dict[col] = qty_median

    # Fill deviation features
    deviation_cols = [col for col in ml_df.columns if 'deviation' in col]
    for col in deviation_cols:
        fill_dict[col] = 0.0

    # Fill diff features
    diff_cols = [col for col in ml_df.columns if 'diff' in col]
    for col in diff_cols:
        fill_dict[col] = 0.0

    # Fill volatility features
    volatility_cols = [col for col in ml_df.columns if 'volatility' in col]
    for col in volatility_cols:
        fill_dict[col] = 0.0

    # Apply all fills at once
    ml_df.fillna(fill_dict, inplace=True)

    # Fill any remaining NaN values
    remaining_nan_cols = ml_df.columns[ml_df.isnull().any()].tolist()
    if remaining_nan_cols:
        print(f"    Filling remaining {len(remaining_nan_cols)} columns...")
        for col in remaining_nan_cols:
            if ml_df[col].dtype in ['int64', 'float64']:
                ml_df[col] = ml_df[col].fillna(ml_df[col].median())
            else:
                ml_df[col] = ml_df[col].fillna(
                    ml_df[col].mode()[0] if not ml_df[col].mode().empty else 0)

    print("  COMPLETE leakage-safe feature engineering completed!")
    print(f"  Total features created: {ml_df.shape[1]}")

    return ml_df

# Apply feature engineering
ml_df = create_ml_features_leakage_safe(main_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df.columns if any(x in col for x in ['market_share', 'days_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df.isnull().sum()
missing_percentage = (missing_values / len(ml_df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0].head(10))



2. COMPREHENSIVE FEATURE ENGINEERING
--------------------------------------------------
Creating comprehensive features with COMPLETE leakage-safe engineering...
Dropped 0 column(s) with ≥50% NaN values:
No columns dropped.
  Data points per branch: {'BLR': np.int64(59), 'COK': np.int64(50), 'MAA': np.int64(59), 'SBD': np.int64(59), 'VJW': np.int64(59)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating additional enhancement features...
  Creating leakage-safe seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating leakage-safe statistical features...
  Creating leakage-safe business features...
  Handling remaining NaN values...
    Filling remaining 6 columns...
  COMPLETE leakage-safe feature engineering completed!
  Total features created: 95

Feature Categories:
  Time-based: 18 fea

In [44]:
def fit_label_encoder_on_train(train_data, test_data, val_data=None):
    """
    Fit LabelEncoder only on training data to avoid leakage
    """
    print("Fitting LabelEncoder on training data only...")

    # Fit encoder on training data
    branch_encoder = LabelEncoder()
    train_data.loc[:, 'branch_encoded'] = branch_encoder.fit_transform(
        train_data['Branch'])

    # Transform test and validation data using fitted encoder
    test_data.loc[:, 'branch_encoded'] = branch_encoder.transform(
        test_data['Branch'])
    if val_data is not None:
        val_data.loc[:, 'branch_encoded'] = branch_encoder.transform(
            val_data['Branch'])

    return train_data, test_data, val_data, branch_encoder

In [45]:
# =============================================================================
# 3. DATA PREPARATION AND TRAIN-TEST SPLIT
# =============================================================================

print("\n3. DATA PREPARATION AND TRAIN-TEST SPLIT")

def prepare_ml_data(df, target_col='Qty', test_size=0.2, validation_size=0.1):
    """
    Prepare data for machine learning models with improved NaN handling
    """
    print("Preparing data for ML models...")
    
    
    # Remove rows with missing target values
    df_clean = df.dropna(subset=[target_col]).copy()
    print(f"  After removing missing target values: {len(df_clean)} samples")
    
    # Check for any remaining missing values
    missing_before = df_clean.isnull().sum().sum()
    print(f"  Missing values before cleaning: {missing_before}")
    
    # Fill any remaining missing values with appropriate strategies
    print("  Filling remaining missing values...")
    
    # Fill numerical columns with median
    numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if col != target_col and df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            print(f"    Filled {col} with median: {df_clean[col].median():.2f}")
    
    # Fill categorical columns with mode
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().any():
            mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f"    Filled {col} with mode: {mode_val}")
    
    # Define feature columns (exclude target and non-predictive columns)
    exclude_cols = [target_col, 'Date', 'Year', 'Month', 'Week', 'Branch', 'Segment', 'Rating']
    feature_cols = [col for col in df_clean.columns if col not in exclude_cols]
    
    print(f"  Total features: {len(feature_cols)}")
    print(f"  Total samples: {len(df_clean)}")
    
    # Check final missing values
    missing_after = df_clean.isnull().sum().sum()
    print(f"  Missing values after cleaning: {missing_after}")
    
    if missing_after > 0:
        print("  Warning: Still have missing values!")
        missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()
        print(f"  Columns with missing values: {missing_cols}")
    
    # Sort by date
    df_clean_sorted = df_clean.sort_values('Date')
    
    # Calculate split indices
    total_samples = len(df_clean_sorted)
    test_start_idx = int(total_samples * (1 - test_size))
    val_start_idx = int(total_samples * (1 - test_size - validation_size))
    # Create splits
    train_data = df_clean_sorted.iloc[:val_start_idx]
    val_data = df_clean_sorted.iloc[val_start_idx:test_start_idx]
    test_data = df_clean_sorted.iloc[test_start_idx:]
    train_data, val_data, test_data, branch_encoder = fit_label_encoder_on_train(train_data, val_data, test_data)
    
    
    # Extract features and targets for each split
    X_train = train_data[feature_cols]
    y_train = train_data[target_col]
    X_val = val_data[feature_cols]
    y_val = val_data[target_col]
    X_test = test_data[feature_cols]
    y_test = test_data[target_col]
    
    print(f"  Train set: {len(X_train)} samples ({len(X_train)/total_samples*100:.1f}%)")
    print(f"  Validation set: {len(X_val)} samples ({len(X_val)/total_samples*100:.1f}%)")
    print(f"  Test set: {len(X_test)} samples ({len(X_test)/total_samples*100:.1f}%)")
    
    # Date ranges for each split
    print(f"  Train period: {train_data['Date'].min()} to {train_data['Date'].max()}")
    print(f"  Validation period: {val_data['Date'].min()} to {val_data['Date'].max()}")
    print(f"  Test period: {test_data['Date'].min()} to {test_data['Date'].max()}")
    
    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'feature_cols': feature_cols,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data,
        'branch_encoder': branch_encoder
    }

# Prepare the data
ml_data = prepare_ml_data(ml_df)

# Display data summary
print(f"\nData Summary:")
print(f"  Features: {len(ml_data['feature_cols'])}")
print(f"  Training samples: {len(ml_data['X_train'])}")
print(f"  Validation samples: {len(ml_data['X_val'])}")
print(f"  Test samples: {len(ml_data['X_test'])}")

# Check for any remaining missing values
print(f"\nMissing Values Check:")
print(f"  Train set missing values: {ml_data['X_train'].isnull().sum().sum()}")
print(f"  Validation set missing values: {ml_data['X_val'].isnull().sum().sum()}")
print(f"  Test set missing values: {ml_data['X_test'].isnull().sum().sum()}")

# Display feature importance preview (using correlation with target)
print(f"\nTop 10 Features by Correlation with Target:")
correlations = ml_data['X_train'].corrwith(ml_data['y_train']).abs().sort_values(ascending=False)
print(correlations.head(10))

# Save the prepared data for later use
print(f"\nSaving prepared data...")
ml_data['X_train'].to_csv('../data/ml_X_train.csv', index=False)
ml_data['X_val'].to_csv('../data/ml_X_val.csv', index=False)
ml_data['X_test'].to_csv('../data/ml_X_test.csv', index=False)
ml_data['y_train'].to_csv('../data/ml_y_train.csv', index=False)
ml_data['y_val'].to_csv('../data/ml_y_val.csv', index=False)
ml_data['y_test'].to_csv('../data/ml_y_test.csv', index=False)

print("Data preparation completed successfully!")



3. DATA PREPARATION AND TRAIN-TEST SPLIT
Preparing data for ML models...
  After removing missing target values: 286 samples
  Missing values before cleaning: 0
  Filling remaining missing values...
  Total features: 92
  Total samples: 286
  Missing values after cleaning: 0
Fitting LabelEncoder on training data only...
  Train set: 200 samples (69.9%)
  Validation set: 28 samples (9.8%)
  Test set: 58 samples (20.3%)
  Train period: 2019-04-01 00:00:00 to 2022-10-01 00:00:00
  Validation period: 2022-10-01 00:00:00 to 2023-04-01 00:00:00
  Test period: 2023-04-01 00:00:00 to 2024-03-01 00:00:00

Data Summary:
  Features: 92
  Training samples: 200
  Validation samples: 28
  Test samples: 58

Missing Values Check:
  Train set missing values: 0
  Validation set missing values: 0
  Test set missing values: 0

Top 10 Features by Correlation with Target:
qty_rolling_mean_2m    0.862004
qty_rolling_sum_2m     0.848419
qty_rolling_min_2m     0.824894
qty_rolling_max_2m     0.800130
qty_roll

In [49]:
# =============================================================================
# 4. LIGHTGBM MODEL IMPLEMENTATION
# =============================================================================

print("\n4. LIGHTGBM MODEL IMPLEMENTATION")
print("-" * 50)

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

def calculate_metrics(y_true, y_pred, model_name="Model"):
    """Calculate comprehensive evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

def train_lightgbm_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train LightGBM model with hyperparameter tuning"""
    
    print("Training LightGBM model...")
    
    # 1. Baseline LightGBM Model
    print("  Training baseline LightGBM model...")
    start_time = time.time()
    
    baseline_params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42
    }
    
    # Create datasets
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # Train baseline model
    baseline_model = lgb.train(
        baseline_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "LightGBM Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "LightGBM Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "LightGBM Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'num_leaves': [31, 50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'feature_fraction': [0.8, 0.9, 0.95, 1.0],
        'bagging_fraction': [0.8, 0.9, 0.95, 1.0],
        'bagging_freq': [5, 10, 15],
        'min_child_samples': [20, 30, 50],
        'reg_alpha': [0, 0.1, 0.5, 1.0],
        'reg_lambda': [0, 0.1, 0.5, 1.0]
    }
    
    # Use RandomizedSearchCV for efficiency
    lgb_model = lgb.LGBMRegressor(
        objective='regression',
        metric='mae',
        boosting_type='gbdt',
        verbose=-1,
        random_state=42,
        n_estimators=1000
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        lgb_model,
        param_grid,
        n_iter=50,  # Number of parameter settings sampled
        cv=3,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = lgb.train(
        final_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "LightGBM Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "LightGBM Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "LightGBM Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importance(importance_type='gain')
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time
        }
    }

# Train LightGBM model
lgb_results = train_lightgbm_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nLightGBM model training completed successfully!")
print(f"Total training time: {sum(lgb_results['training_times'].values()):.2f} seconds")



4. LIGHTGBM MODEL IMPLEMENTATION
--------------------------------------------------
Training LightGBM model...
  Training baseline LightGBM model...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[446]	valid_0's l1: 1080.09
    Baseline training time: 0.33 seconds
    Baseline validation RMSE: 1697.1306
    Baseline validation R²: 0.8759
  Performing hyperparameter tuning...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[246]	valid_0's l1: 993.138
    Best parameters: {'reg_lambda': 1.0, 'reg_alpha': 0, 'num_leaves': 100, 'min_child_samples': 20, 'learning_rate': 0.1, 'feature_fraction': 0.95, 'bagging_freq': 15, 'bagging_fraction': 0.95}
    Tuning time: 15.43 seconds
  Training final model with best parameters...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[246]	valid_0's l1: 993.138
    Final training time: 0.14 seconds
    Final 

In [50]:
# =============================================================================
# 5. XGBOOST MODEL IMPLEMENTATION
# =============================================================================

print("\n5. XGBOOST MODEL IMPLEMENTATION")
print("-" * 50)

def train_xgboost_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train XGBoost model with hyperparameter tuning"""
    
    print("Training XGBoost model...")
    
    # 1. Baseline XGBoost Model
    print("  Training baseline XGBoost model...")
    start_time = time.time()
    
    baseline_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 1000,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'verbosity': 0
    }
    
    # Train baseline model
    baseline_model = xgb.XGBRegressor(**baseline_params)
    baseline_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "XGBoost Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "XGBoost Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "XGBoost Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'max_depth': [3, 4, 5, 6, 7, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'n_estimators': [500, 800, 1000, 1200],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bynode': [0.6, 0.7, 0.8, 0.9, 1.0],
        'reg_alpha': [0, 0.1, 0.5, 1.0],
        'reg_lambda': [0, 0.1, 0.5, 1.0, 2.0],
        'gamma': [0, 0.1, 0.5, 1.0],
        'min_child_weight': [1, 3, 5, 7]
    }
    
    # Use RandomizedSearchCV for efficiency
    xgb_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        eval_metric='mae',
        random_state=42,
        verbosity=0
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        xgb_model,
        param_grid,
        n_iter=50,  # Number of parameter settings sampled
        cv=3,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=[(X_val, y_val)],
                     verbose=False)
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = xgb.XGBRegressor(**final_params)
    final_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "XGBoost Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "XGBoost Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "XGBoost Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importances_
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    # 6. Advanced XGBoost Features
    print("  Training advanced XGBoost model with additional features...")
    start_time = time.time()
    
    # Advanced parameters for better performance
    advanced_params = final_params.copy()
    advanced_params.update({
        'tree_method': 'hist',  # Use histogram-based algorithm
        'grow_policy': 'lossguide',  # Grow policy for better performance
        'max_leaves': 0,  # Let max_depth control tree size
        'max_bin': 256,  # Number of bins for histogram
        'predictor': 'cpu_predictor',  # Use CPU predictor
        'enable_categorical': False,  # Disable categorical features
        'interaction_constraints': None,  # No interaction constraints
        'monotone_constraints': None,  # No monotone constraints
    })
    
    # Train advanced model
    advanced_model = xgb.XGBRegressor(**advanced_params)
    advanced_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    advanced_time = time.time() - start_time
    
    # Make predictions with advanced model
    advanced_train_pred = advanced_model.predict(X_train)
    advanced_val_pred = advanced_model.predict(X_val)
    advanced_test_pred = advanced_model.predict(X_test)
    
    # Calculate advanced metrics
    advanced_train_metrics = calculate_metrics(y_train, advanced_train_pred, "XGBoost Advanced Train")
    advanced_val_metrics = calculate_metrics(y_val, advanced_val_pred, "XGBoost Advanced Val")
    advanced_test_metrics = calculate_metrics(y_test, advanced_test_pred, "XGBoost Advanced Test")
    
    print(f"    Advanced training time: {advanced_time:.2f} seconds")
    print(f"    Advanced validation RMSE: {advanced_val_metrics['RMSE']:.4f}")
    print(f"    Advanced validation R²: {advanced_val_metrics['R2']:.4f}")
    
    # Compare all XGBoost models
    print("  All XGBoost models comparison:")
    print(f"    Baseline RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Final RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Advanced RMSE: {advanced_val_metrics['RMSE']:.4f}")
    
    best_model = 'Advanced' if advanced_val_metrics['RMSE'] < final_val_metrics['RMSE'] else 'Final'
    print(f"    Best XGBoost model: {best_model}")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'advanced_model': advanced_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'advanced_metrics': {
            'train': advanced_train_metrics,
            'val': advanced_val_metrics,
            'test': advanced_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            },
            'advanced': {
                'train': advanced_train_pred,
                'val': advanced_val_pred,
                'test': advanced_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time,
            'advanced': advanced_time
        }
    }

# Train XGBoost model
xgb_results = train_xgboost_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nXGBoost model training completed successfully!")
print(f"Total training time: {sum(xgb_results['training_times'].values()):.2f} seconds")



5. XGBOOST MODEL IMPLEMENTATION
--------------------------------------------------
Training XGBoost model...
  Training baseline XGBoost model...
    Baseline training time: 2.98 seconds
    Baseline validation RMSE: 721.8355
    Baseline validation R²: 0.9776
  Performing hyperparameter tuning...
    Best parameters: {'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 1000, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.15, 'gamma': 1.0, 'colsample_bytree': 1.0, 'colsample_bynode': 0.6, 'colsample_bylevel': 0.7}
    Tuning time: 29.30 seconds
  Training final model with best parameters...
    Final training time: 1.31 seconds
    Final validation RMSE: 1152.7792
    Final validation R²: 0.9428
  Analyzing feature importance...
    Top 10 most important features:
      1. qty_rolling_min_2m: 0.2752
      2. qty_rolling_mean_2m: 0.1566
      3. qty_diff_1m: 0.0941
      4. Max Humidity: 0.0696
      5. qty_rolling_mean_3m: 0.0688
      6. qty_rolling_min_

In [51]:
# =============================================================================
# 6. CATBOOST MODEL IMPLEMENTATION
# =============================================================================

print("\n6. CATBOOST MODEL IMPLEMENTATION")
print("-" * 50)

def train_catboost_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train CatBoost model with hyperparameter tuning"""
    
    print("Training CatBoost model...")
    
    # 1. Baseline CatBoost Model
    print("  Training baseline CatBoost model...")
    start_time = time.time()
    
    baseline_params = {
        'iterations': 1000,
        'learning_rate': 0.1,
        'depth': 6,
        'l2_leaf_reg': 3,
        'bootstrap_type': 'Bayesian',
        'random_seed': 42,
        'od_type': 'Iter',
        'od_wait': 100,
        'verbose': False
    }
    
    # Train baseline model
    baseline_model = cb.CatBoostRegressor(**baseline_params)
    baseline_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "CatBoost Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "CatBoost Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "CatBoost Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'iterations': [500, 800, 1000, 1200],
        'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'depth': [4, 5, 6, 7, 8],
        'l2_leaf_reg': [1, 3, 5, 7, 9],
        'bootstrap_type': ['Bayesian', 'Bernoulli'],
        'bagging_temperature': [0, 0.5, 1.0],
        'random_strength': [0, 1, 2],
        'one_hot_max_size': [2, 10, 20],
        'leaf_estimation_method': ['Newton', 'Gradient'],
        'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide']
    }
    
    # Use RandomizedSearchCV for efficiency
    cb_model = cb.CatBoostRegressor(
        random_seed=42,
        od_type='Iter',
        od_wait=100,
        verbose=False
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        cb_model,
        param_grid,
        n_iter=30,  # Number of parameter settings sampled
        cv=3,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=(X_val, y_val),
                     early_stopping_rounds=100,
                     verbose=False)
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = cb.CatBoostRegressor(**final_params)
    final_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "CatBoost Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "CatBoost Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "CatBoost Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importances_
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time
        }
    }

# Train CatBoost model
cb_results = train_catboost_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nCatBoost model training completed successfully!")
print(f"Total training time: {sum(cb_results['training_times'].values()):.2f} seconds")



6. CATBOOST MODEL IMPLEMENTATION
--------------------------------------------------
Training CatBoost model...
  Training baseline CatBoost model...
    Baseline training time: 7.00 seconds
    Baseline validation RMSE: 1728.1576
    Baseline validation R²: 0.8714
  Performing hyperparameter tuning...
    Best parameters: {'random_strength': 0, 'one_hot_max_size': 2, 'learning_rate': 0.1, 'leaf_estimation_method': 'Newton', 'l2_leaf_reg': 1, 'iterations': 800, 'grow_policy': 'Depthwise', 'depth': 4, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1.0}
    Tuning time: 137.86 seconds
  Training final model with best parameters...
    Final training time: 1.40 seconds
    Final validation RMSE: 567.2602
    Final validation R²: 0.9861
  Analyzing feature importance...
    Top 10 most important features:
      1. qty_rolling_min_2m: 24.1686
      2. qty_rolling_mean_2m: 22.9840
      3. qty_diff_1m: 22.2370
      4. qty_rolling_max_2m: 9.8048
      5. qty_diff_3m: 7.4520
      6. qt

In [ ]:
# =============================================================================
# 7. FORCAST NEXT 12 MONTHS
# =============================================================================


last_date = main_df['Date'].max()
branches = main_df['Branch'].unique()
print("Last available month:", last_date)
print("Branches:", branches)

future_dates = pd.date_range(
    start=last_date + pd.offsets.MonthBegin(),
    periods=12, freq='MS'
)


future_df = pd.DataFrame([(d, b) for d in future_dates for b in branches],
                         columns=['Date', 'Branch'])
future_df = future_df.merge(weather_data, on=['Date', 'Branch'], how='left')
future_df = future_df.merge(trends_df, on=['Date'], how='left')

# Fill missing future external data with mean or last available value
for col in ['Avg Temp', 'Avg Humidity', 'Avg Wind Speed', 'Interest']:
    if col in future_df.columns:
        future_df[col] = future_df[col].fillna(main_df[col].mean())

# Add seasonality level
future_df['Seasonality_Level'] = future_df['Date'].dt.month_name().str[:3].map({
    'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
    'May': 2, 'Jan': 2,
    'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
})

# Append future_df to history
combined_df = pd.concat([main_df, future_df], ignore_index=True)
combined_features = create_ml_features_leakage_safe(combined_df)

# Keep only the new future months
future_features = combined_features[combined_features['Date'].isin(future_dates)].copy().reset_index(drop=True)
X_future = future_features[ml_data['feature_cols']].fillna(0)

Last available month: 2024-03-01 00:00:00
Branches: ['BLR' 'MAA' 'SBD' 'VJW' 'COK']


In [76]:
future_features

,Date,Branch,Qty,Min Temp,Max Temp,Avg Temp,Min Humidity,Max Humidity,Avg Humidity,Min Wind Speed,...,humidity_range,wind_range,temp_monthly_mean_past,temp_deviation,humidity_monthly_mean_past,humidity_deviation,months_since_last_sale,sales_momentum_3m,sales_momentum_6m,prev_market_share
0,2024-04-01,BLR,2909.75,20.0,37.0,28.940000,11.0,94.0,42.493056,1.8,...,83.0,25.9,27.041071,1.898929,54.193651,-11.700596,1.018397,2.487098,1.908966,0.092917
1,2024-05-01,BLR,2909.75,20.0,37.0,26.668011,18.0,100.0,67.383065,1.8,...,82.0,42.8,26.270377,0.397634,69.900511,-2.517447,0.985545,3.180794,2.091337,0.000000
2,2024-06-01,BLR,2909.75,20.0,33.0,24.359722,46.0,100.0,76.313889,3.6,...,54.0,33.5,24.810296,-0.450574,76.857086,-0.543197,1.018397,1.070312,2.300582,0.000000
3,2024-07-01,BLR,2909.75,20.0,31.0,23.513441,51.0,94.0,78.372312,7.6,...,43.0,33.4,23.654283,-0.140842,82.423583,-4.051271,0.985545,1.070312,2.487098,0.000000
4,2024-08-01,BLR,2909.75,20.0,31.0,23.981183,49.0,100.0,79.168011,1.8,...,51.0,37.1,23.694694,0.286488,81.331796,-2.163786,1.018397,1.070312,3.180794,0.000000
5,2024-09-01,BLR,2909.75,19.0,30.0,23.947917,37.0,95.0,74.143056,3.6,...,58.0,37.1,23.512979,0.434937,83.298177,-9.155121,1.018397,1.070312,1.087899,0.000000
6,2024-10-01,BLR,2909.75,18.0,31.0,23.466398,37.0,100.0,79.661290,1.8,...,63.0,25.9,23.214059,0.252339,82.075963,-2.414672,0.985545,1.070312,1.087899,0.000000
7,2024-11-01,BLR,2909.75,16.0,28.0,21.892639,36.0,100.0,77.916667,3.6,...,64.0,24.1,21.905343,-0.012704,84.473688,-6.557021,1.018397,1.070312,1.087899,0.000000
8,2024-12-01,BLR,2909.75,14.0,28.0,21.342742,35.0,100.0,80.017473,3.6,...,65.0,22.3,20.831787,0.510955,81.143211,-1.125738,0.985545,1.070312,1.087899,0.000000
9,2025-01-01,BLR,2909.75,12.0,31.0,20.597581,13.0,100.0,68.731183,1.8,...,87.0,24.1,21.293069,-0.695489,70.952641,-2.221459,1.018397,1.070312,1.087899,0.000000


In [78]:
# =============================================================================
# 8. PREDICTIONS
# =============================================================================
cat_preds = cb_results['final_model'].predict(X_future)
xgb_preds = xgb_results['final_model'].predict(X_future)
lgb_preds = lgb_results['final_model'].predict(X_future)

results = {
    'Date': future_dates,
    'Branch': branches,
    'CatBoost': cat_preds,
    'CatBoost_sum': cat_preds.sum(),
    'XGBoost': xgb_preds,
    'XGBoost_sum': xgb_preds.sum(),
    'LightGBM': lgb_preds,
    'LightGBM_sum': lgb_preds.sum()
}

print(results['CatBoost_sum'])
print(results['XGBoost_sum'])
print(results['LightGBM_sum'])

202067.1020254322
205532.69
229280.8135124142


In [79]:
current_df = pd.read_csv('../data/Final Sales.csv')

In [88]:
current_df[(current_df['Date'] >= '2022-04-01') & (current_df['Date'] < '2023-04-01')].groupby('Branch').agg({'Sales Qty.': 'sum'}).reset_index()

,Branch,Sales Qty.
0,BANGALORE,28855.0
1,CHENNAI,101536.0
2,COCHIN,20290.0
3,HYDERABAD,82564.0
4,VIJAYAWADA,34702.0


In [82]:
current_df[current_df['Date'] >= '2024-04-01'].groupby('Branch').agg({'Sales Qty.': 'sum'}).reset_index()

,Branch,Sales Qty.
0,BANGALORE,57489.0
1,CHENNAI,212879.0
2,COCHIN,79464.0
3,HYDERABAD,149219.0
4,VIJAYAWADA,109268.0


In [87]:
results = []

base = 0
for j in range(5):
    curr = 0
    base =j*12 
    for i in range(12):
        curr += xgb_preds[base + i]
    results.append(curr)
results

[np.float32(39625.383),
 np.float32(40157.137),
 np.float32(41232.066),
 np.float32(42782.46),
 np.float32(41735.637)]